# 01 — Exploratory Data Analysis
## Smart Irrigation for Tomato (easy walkthrough)

This notebook reads **3,000 Kathmandu field logs** from the ESP32 (DHT11 + soil probe + pump) and answers, in order:

1. **Is the data trustworthy?** (missing values, realistic Kathmandu weather)
2. **What is each sensor telling us?** (soil, temperature, humidity, pressure)
3. **What was the old pump doing?** (almost a soil on/off switch)
4. **How should a tomato crop be watered instead?** (FAO-56: soil + heat + dry air)
5. **Where do those two policies disagree?** (that disagreement is the FYP)

You do not need to remember formulas to follow the plots. Each section ends with a short **Takeaway**.


## Dictionary — what each column means

| Column | In plain English | Typical healthy range here |
| --- | --- | --- |
| `soilMoisture` | How wet the root zone is (0% bone dry, 100% saturated) | Tomatoes often want roughly 55–75% |
| `temperature` | Air temperature from the DHT11 | Tomato comfort ~18–27 °C; hotter plants drink more |
| `humidity` | How much water is already in the air | Low humidity = air “pulls” water from leaves |
| `pressure` | Air pressure (hPa) | Kathmandu ~850 hPa, **not** 1013 at sea level |
| `vpd_kpa` | Vapor pressure deficit — “how thirsty is the air?” | Higher VPD → crop uses water faster |
| `heat_stress` | 0 if cool, up to 1 if well above 27 °C | Extra irrigation pressure on hot days |
| `dry_hot_index` | Dry soil **and** thirsty air together | Strong “please water” signal |
| `pump_historical` | What the old controller actually did | 1 = pump ON, 0 = OFF |
| `irrigate` | Tomato FAO label we train the API on | 1 = should irrigate |

**VPD in one sentence:** hot + dry air has high VPD, so the plant loses water even if the soil is only a bit dry.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.preprocess import PROCESSED_CSV, main as preprocess_main
from src.generate_season import OUT_CSV as SEASON_CSV, main as generate_season_main
from src.eda import run_eda

sns.set_theme(style="whitegrid", context="notebook")
if not PROCESSED_CSV.exists():
    preprocess_main()
if not SEASON_CSV.exists():
    generate_season_main()

df = pd.read_csv(PROCESSED_CSV)
season = pd.read_csv(SEASON_CSV, parse_dates=["timestamp"])
FIG = ROOT / "results" / "figures"

def show_fig(name, caption):
    path = FIG / name
    if path.exists():
        display(Markdown(f"**{caption}**"))
        display(Image(filename=str(path)))
    else:
        print("Missing", path, "— run the last cell (`run_eda()`) first.")

print(f"Loaded {len(df):,} field rows and {len(season):,} simulated 15-min season rows.")
df[["temperature", "humidity", "soilMoisture", "pressure", "vpd_kpa", "pump_historical", "irrigate"]].head()


## 1. Is the data clean?

Before looking at irrigation, check that the sensors are not broken.

- **3,000 rows**, all complete (no missing cells).
- Pressure around **845–865 hPa** is what you expect in Kathmandu (~1,400 m), not 1013 hPa at the coast.
- Temperature **18–39 °C** and humidity **38–81%** are plausible spring/summer greenhouse or field days.

**Takeaway:** little cleaning is needed. We can trust the logs for modeling.


In [ ]:
quality = pd.DataFrame({
    "missing": df.isna().sum(),
    "min": df.select_dtypes("number").min(),
    "median": df.select_dtypes("number").median(),
    "max": df.select_dtypes("number").max(),
})
print("rows, columns:", df.shape)
quality.loc[["temperature", "humidity", "pressure", "soilMoisture", "vpd_kpa", "pump_historical", "irrigate"]].round(2)


In [ ]:
show_fig("01_sensor_distributions.png", "Figure — how the four raw sensors are spread")


## 2. Simple picture: irrigate vs don’t irrigate

Forget models for a moment. Compare **average** readings when the tomato rule says water vs when it says wait.

| When we irrigate, soil is… | When we don’t, soil is… |
| --- | --- |
| Drier | Wetter |
| Air a bit warmer / thirstier (higher VPD) | Air milder |

**Takeaway:** soil moisture is the main switch. Climate (heat, dry air) nudges the decision.


In [ ]:
print("Average sensors by tomato label:\n")
print(
    df.groupby(df["irrigate"].map({0: "Don't irrigate", 1: "Irrigate"}))[
        ["soilMoisture", "temperature", "humidity", "vpd_kpa"]
    ].mean().round(2)
)
show_fig("15_mean_by_label.png", "Figure — four averages, irrigate vs don't")


## 3. Put soil into easy buckets

Percentages are hard to remember. Bins are not:

| Bucket | Moisture | What it means for tomato |
| --- | --- | --- |
| Dry | 0–30% | Water now |
| Low | 30–55% | Usually water (FAO start band is near 55%) |
| OK | 55–75% | Often wait; heat/VPD may still ask for water |
| Wet | 75–100% | Do not irrigate (risk of waterlogging) |

In this dataset: **dry = 100% irrigate**, **wet = 0% irrigate**, **low ≈ 97%**, **OK ≈ 31%**.

**Takeaway:** a single 55% cutoff is a decent baseline, but the “OK” band is where climate should matter.


In [ ]:
bands = pd.cut(
    df["soilMoisture"],
    bins=[-0.1, 30, 55, 75, 100.1],
    labels=["Dry (0–30%)", "Low (30–55%)", "OK (55–75%)", "Wet (75–100%)"],
)
print(df.groupby(bands, observed=False)["irrigate"].agg(rows="size", irrigate_rate="mean").round(3))
show_fig("16_irrigate_rate_by_soil_bin.png", "Figure — irrigation rate by soil bucket")
show_fig("09_soil_moisture_kde.png", "Figure — dry vs wet overlap (dashed line = 55%)")


## 4. What was the old pump doing?

The logged pump is **ON 52.3%** of the time.

Correlation with soil moisture is about **−0.85** (wetter soil → pump OFF). Temperature, humidity, and pressure are almost **uncorrelated** with the pump.

That matches the ESP32 firmware: relay ON when the probe looks dry (`soilMoisture <= 0` on the raw dry stop), not when the tomato is heat-stressed.

**Takeaway:** cloning the historical pump with ML is easy and not the project goal. The baseline to beat is “if soil < 55%, irrigate”.


In [ ]:
print("Pump ON rate:", round(df["pump_historical"].mean(), 3))
print("Tomato irrigate rate:", round(df["irrigate"].mean(), 3))
print("How often they match:", round((df["pump_historical"] == df["irrigate"]).mean(), 3))
print()
print("Correlation with the old pump (near 0 means the pump ignored that sensor):")
print(
    df[["soilMoisture", "temperature", "humidity", "pressure", "vpd_kpa", "pump_historical"]]
    .corr()["pump_historical"]
    .drop("pump_historical")
    .round(3)
)
show_fig("06_class_balance.png", "Figure — how often pump ON vs tomato irrigate")
show_fig("04_soil_temp_decision_scatter.png", "Figure — same sensors, two policies")


## 5. Tomato rule (FAO-56) in plain language

FAO tomato **management allowed depletion ≈ 0.40**. In relative moisture that means: start watering near **55–60%**, not when the soil is empty.

Then adjust:

- **Hot or dry air (high VPD)** → irrigate a bit sooner (threshold goes up).
- **Cool, humid air** → you can wait.
- **Already waterlogged** → never irrigate.

The tomato label (`irrigate`) is ON **58.3%** of rows — a little more often than the old pump — because it waters earlier on stressful days.

**Takeaway:** train the API on `irrigate`, not `pump_historical`.


In [ ]:
show_fig("05_soil_vpd_scatter.png", "Figure — dry soil + thirsty air (high VPD) → irrigate")
show_fig("22_temp_humidity_comfort.png", "Figure — tomato comfort band 18–27 °C (green tint)")
show_fig("23_soil_temp_irrigate_heatmap.png", "Figure — read like a table: dry+hot = water, wet = don't")


## 6. Where the old pump and the tomato rule disagree

They give the **same ON/OFF on 92.2%** of rows. The remaining **7.8%** is the FYP:

| Situation | Count | Meaning |
| --- | --- | --- |
| Pump skipped, tomato says water | **207** | Heat / dry air — crop needed water earlier |
| Pump watered, tomato says skip | **26** | Pump ran when the FAO rule would wait |

**Takeaway:** the new system mostly *adds* irrigation on stressful climate days, and rarely *removes* a pump event.


In [ ]:
print(pd.crosstab(
    df["pump_historical"].map({0: "Pump OFF", 1: "Pump ON"}),
    df["irrigate"].map({0: "Tomato: don't", 1: "Tomato: irrigate"}),
    margins=True,
))
show_fig("17_pump_vs_tomato_agreement.png", "Figure — 2×2 agreement table")
show_fig("18_label_disagreements.png", "Figure — only the mismatch rows (soil vs VPD)")


## 7. Which signals matter? (easier than a full heatmap)

A correlation of **−1 or +1** is a strong link; **0** means “no linear link”.

- **Soil moisture** is strongly negative for both pump and tomato label (wet → don’t water).
- **VPD, ET0, heat, dry-hot index** move with the tomato label, but sit near **zero for the pump**.
- **Pressure** does almost nothing — the firmware always sends 0 anyway; the API fills Kathmandu’s mean.

**Takeaway:** give the model FAO features so it can see climate, not only soil.


In [ ]:
show_fig("19_what_predicts_irrigation.png", "Figure — grey = old pump, blue = tomato label")
show_fig("20_engineered_features.png", "Figure — VPD, heat stress, dry-hot index")
show_fig("03_correlation_heatmap.png", "Figure — full Pearson heatmap (optional detail)")


## 8. Time of day and crop stage (simulated season)

The Excel workbook has **no timestamps**, so we cannot plot a real week from the 3,000 logs.

For **diurnal cycle and growth stages only**, we simulate a Kathmandu spring tomato crop at the same **15-minute** upload interval, using FAO crop coefficients (Kc 0.60 → 1.15 → 0.80).

**This file is for EDA, not for training.** The production model still uses the 3,000 real logs.

**Takeaway:** morning irrigation (simulated) wastes less water than midday. Mid-season (highest Kc) needs water most often.


In [ ]:
print("Irrigation rate by FAO growth stage:")
print(season.groupby("growth_stage", observed=False)["irrigate"].mean().round(3))
show_fig("07_season_week_timeseries.png", "Figure — one simulated week (temp, soil, irrigate)")
show_fig("08_irrigation_by_growth_stage.png", "Figure — how often we irrigate in each crop stage")
show_fig("21_hourly_irrigation_season.png", "Figure — irrigation by hour of day (simulated)")


## 9. What this means for the ML models

| Finding | What we do next |
| --- | --- |
| Data is complete and physically plausible | No heavy cleaning |
| Old pump ≈ soil switch | Baseline = irrigate if soil < 55% |
| Tomato label uses soil × climate | Target column = `irrigate` |
| They disagree on 7.8% of rows (mostly extra watering in heat) | That is the behaviour we want the API to learn |
| No timestamps on the 3,000 logs | Stratified 80/20 split (not a time split) |
| Pressure is unused / often 0 on the ESP32 | Impute Kathmandu mean at inference |

Next notebook: `02_model_training.ipynb` compares **XGBoost, Random Forest, and SVM** on this tomato label.

To rebuild every EDA figure under `results/figures/`, run the cell below.


In [ ]:
# Regenerates figures 01–09 and 15–23. Safe to re-run.
summary = run_eda()
summary["plain_language"]
